# RAG Retrieval Comparison

Vector RAG, Keyword RAG (BM25), Hybrid RAG, and Reranked RAG. Original code and document collection retained.

In [ ]:
!pip -q install sentence-transformers rank_bm25 scikit-learn

"""
RAG retrieval comparison:
Vector RAG, Keyword RAG (BM25), Hybrid RAG, and Reranked RAG.
"""

import re
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.metrics.pairwise import cosine_similarity


# ---------------------------------------------------------
# Small document collection
# ---------------------------------------------------------

documents = [
    {
        "id": "doc1",
        "title": "Refund Policy",
        "text": """
        Customers can request a refund within 30 days of purchase.
        The item must be unused and returned in its original packaging.
        Refunds are issued to the original payment method.
        """
    },
    {
        "id": "doc2",
        "title": "Damaged Products",
        "text": """
        Damaged products must be reported within 7 days of delivery.
        Customers should send a photo of the damaged item.
        After approval, the company will offer a replacement or refund.
        """
    },
    {
        "id": "doc3",
        "title": "Shipping Policy",
        "text": """
        Standard shipping takes 3 to 5 business days.
        Express shipping takes 1 to 2 business days.
        International shipping may take longer.
        """
    },
    {
        "id": "doc4",
        "title": "Section 12.4 Termination",
        "text": """
        Section 12.4 states that either party may terminate the agreement
        with 30 days written notice if the other party violates the contract.
        """
    },
    {
        "id": "doc5",
        "title": "Medical Synonyms Example",
        "text": """
        A physician must review the patient record before treatment.
        The doctor should confirm allergies and medication history.
        """
    }
]


# ---------------------------------------------------------
# Method 1: Chunking
# ---------------------------------------------------------

def chunk_text(text, chunk_size=35, overlap=8):
    """
    Splits a document into smaller chunks.

    This is for the Chunking method.
    In real RAG, chunks are often 300 to 800 tokens.
    Here we use small chunks for teaching.
    """

    words = text.split()
    chunks = []

    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)

        start = end - overlap

        if start >= len(words):
            break

    return chunks


def build_chunks(documents):
    """
    Creates chunks with metadata.
    Metadata helps us know where each chunk came from.
    """

    chunks = []

    for doc in documents:
        doc_chunks = chunk_text(doc["text"])

        for i, chunk in enumerate(doc_chunks):
            chunks.append({
                "chunk_id": f"{doc['id']}_chunk_{i}",
                "doc_id": doc["id"],
                "title": doc["title"],
                "text": chunk
            })

    return chunks


chunks = build_chunks(documents)


# ---------------------------------------------------------
# Load embedding and reranker models
# ---------------------------------------------------------

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# CrossEncoder is used for reranking.
# It scores a query and a chunk together.
reranker_model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")


# ---------------------------------------------------------
# Method 2: Embeddings
# ---------------------------------------------------------

def create_embeddings(chunks):
    """
    Converts text chunks into dense vectors.

    This is for the Embeddings method.
    """

    texts = [chunk["text"] for chunk in chunks]
    embeddings = embedding_model.encode(texts, convert_to_numpy=True)
    return embeddings


chunk_embeddings = create_embeddings(chunks)


# ---------------------------------------------------------
# Method 3: Vector RAG
# ---------------------------------------------------------

def vector_search(question, chunks, chunk_embeddings, k=3):
    """
    Retrieves chunks using semantic similarity.

    This is for Vector RAG.
    It compares the question embedding with chunk embeddings.
    """

    question_embedding = embedding_model.encode([question], convert_to_numpy=True)

    scores = cosine_similarity(question_embedding, chunk_embeddings)[0]

    ranked_indices = np.argsort(scores)[::-1][:k]

    results = []
    for idx in ranked_indices:
        result = chunks[idx].copy()
        result["score"] = float(scores[idx])
        result["method"] = "Vector RAG"
        results.append(result)

    return results


# ---------------------------------------------------------
# Method 4: Keyword RAG, BM25
# ---------------------------------------------------------

def tokenize(text):
    """
    Simple tokenizer for BM25 keyword search.
    """
    text = text.lower()
    return re.findall(r"\b\w+\.?\w*\b", text)

tokenized_chunks = [tokenize(chunk["text"]) for chunk in chunks]
bm25 = BM25Okapi(tokenized_chunks)


def keyword_search(question, chunks, k=3):
    """
    Retrieves chunks using BM25 keyword matching.
    This is for Keyword RAG.
    It works well for exact terms, codes, names, and section numbers.
    """
    query_tokens = tokenize(question)
    scores = bm25.get_scores(query_tokens)
    ranked_indices = np.argsort(scores)[::-1][:k]

    results = []
    for idx in ranked_indices:
        result = chunks[idx].copy()
        result["score"] = float(scores[idx])
        result["method"] = "Keyword RAG, BM25"
        results.append(result)

    return results


# ---------------------------------------------------------
# Method 5: Hybrid RAG
# ---------------------------------------------------------

def normalize_scores(scores):
    """
    Normalizes scores into the range 0 to 1.
    """

    scores = np.array(scores, dtype=float)

    if scores.max() == scores.min():
        return np.zeros_like(scores)

    return (scores - scores.min()) / (scores.max() - scores.min())


def hybrid_search(question, chunks, chunk_embeddings, k=3, alpha=0.5):
    """
    Combines vector search and BM25 keyword search.

    This is for Hybrid RAG.
    alpha controls the balance:
    alpha = 0.5 means half vector score, half keyword score.
    """

    question_embedding = embedding_model.encode([question], convert_to_numpy=True)

    vector_scores = cosine_similarity(question_embedding, chunk_embeddings)[0]
    keyword_scores = bm25.get_scores(tokenize(question))

    vector_scores_norm = normalize_scores(vector_scores)
    keyword_scores_norm = normalize_scores(keyword_scores)

    hybrid_scores = alpha * vector_scores_norm + (1 - alpha) * keyword_scores_norm

    ranked_indices = np.argsort(hybrid_scores)[::-1][:k]

    results = []
    for idx in ranked_indices:
        result = chunks[idx].copy()
        result["score"] = float(hybrid_scores[idx])
        result["method"] = "Hybrid RAG"
        results.append(result)

    return results


# ---------------------------------------------------------
# Method 6: Reranked RAG
# ---------------------------------------------------------

def reranked_search(question, chunks, chunk_embeddings, retrieve_k=5, final_k=3):
    """
    First retrieves many chunks with vector search.
    Then reranks them with a cross-encoder.

    This is for Reranked RAG.
    Retriever is fast.
    Reranker is slower but more precise.
    """

    initial_results = vector_search(question, chunks, chunk_embeddings, k=retrieve_k)

    pairs = [(question, result["text"]) for result in initial_results]

    rerank_scores = reranker_model.predict(pairs)

    for result, score in zip(initial_results, rerank_scores):
        result["score"] = float(score)
        result["method"] = "Reranked RAG"

    reranked_results = sorted(
        initial_results,
        key=lambda x: x["score"],
        reverse=True
    )

    return reranked_results[:final_k]



# ---------------------------------------------------------
# Helper function to print results
# ---------------------------------------------------------

def print_results(title, question, results):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)
    print(f"Question: {question}\n")

    for i, result in enumerate(results, start=1):
        print(f"Rank {i}")
        print(f"Method: {result['method']}")
        print(f"Title: {result['title']}")
        print(f"Score: {result['score']:.4f}")
        print(f"Text: {result['text']}")
        print("-" * 80)


In [ ]:
question_1 = "What is the return deadline for damaged products?"

# Vector RAG
vector_results = vector_search(question_1, chunks, chunk_embeddings, k=3)

print_results("Method 3: Vector RAG", question_1, vector_results)

In [ ]:
# Keyword RAG, BM25
keyword_results = keyword_search(question_1, chunks, k=3)
print_results("Method 4: Keyword RAG, BM25", question_1, keyword_results)

In [ ]:
# Hybrid RAG
hybrid_results = hybrid_search(question_1, chunks, chunk_embeddings, k=3, alpha=0.5)
print_results("Method 5: Hybrid RAG", question_1, hybrid_results)

In [ ]:
# Reranked RAG
reranked_results = reranked_search(question_1, chunks, chunk_embeddings, retrieve_k=5, final_k=3)
print_results("Method 6: Reranked RAG", question_1, reranked_results)

# Four RAG Retrieval Methods with Libraries

In [ ]:
# Create the document collection
!pip -q install -U haystack-ai sentence-transformers-haystack

from haystack import Document
from haystack.document_stores.in_memory import InMemoryDocumentStore

policies = [
    ("doc1", "Refund Policy",
     "Customers can request a refund within 30 days of purchase. "
     "The item must be unused and in its original packaging. "
     "Refunds go to the original payment method."),
    ("doc2", "Damaged Products",
     "Damaged products must be reported within 7 days of delivery. "
     "Customers should send a photo of the damaged item. "
     "After approval, the company offers a replacement or refund."),
    ("doc3", "Shipping Policy",
     "Standard shipping takes 3 to 5 business days. "
     "Express shipping takes 1 to 2 business days. International shipping may take longer."),
    ("doc4", "Section 12.4 Termination",
     "Section 12.4 states that either party may terminate the agreement "
     "with 30 days written notice if the other party violates the contract."),
    ("doc5", "Medical Record Review",
     "A physician must review the patient record before treatment. "
     "The doctor should confirm allergies and medication history."),
]

documents = [Document(id=id, content=text, meta={"title": title})
             for id, title, text in policies]

store = InMemoryDocumentStore(embedding_similarity_function="cosine")

In [ ]:
# Load pretrained models and index documents

from haystack.components.retrievers.in_memory import (
    InMemoryBM25Retriever, InMemoryEmbeddingRetriever)
from haystack.components.joiners import DocumentJoiner
from haystack_integrations.components.embedders.sentence_transformers import (
    SentenceTransformersDocumentEmbedder, SentenceTransformersTextEmbedder)
from haystack_integrations.components.rankers.sentence_transformers import (
    SentenceTransformersSimilarityRanker)

MODEL = "sentence-transformers/all-MiniLM-L6-v2"

doc_embedder = SentenceTransformersDocumentEmbedder(model=MODEL)
doc_embedder.warm_up()
store.write_documents(doc_embedder.run(documents)["documents"])

query_embedder = SentenceTransformersTextEmbedder(model=MODEL)
query_embedder.warm_up()

vector_retriever = InMemoryEmbeddingRetriever(document_store=store)
bm25_retriever = InMemoryBM25Retriever(document_store=store)
hybrid_joiner = DocumentJoiner(join_mode="reciprocal_rank_fusion")

reranker = SentenceTransformersSimilarityRanker(
    model="cross-encoder/ms-marco-MiniLM-L-6-v2"
)
reranker.warm_up()


In [ ]:
# Compare four methods

question = "What is the return deadline for damaged products?"
query_embedding = query_embedder.run(text=question)["embedding"]

# 1. Vector retrieval
vector_results = vector_retriever.run(query_embedding=query_embedding, top_k=3)["documents"]

# 2. Keyword retrieval (BM25)
bm25_results = bm25_retriever.run(query=question, top_k=3)["documents"]

# 3. Hybrid retrieval (BM25 + vector, using built-in rank fusion)
hybrid_results = hybrid_joiner.run(documents=[
    bm25_retriever.run(query=question, top_k=5)["documents"],
    vector_retriever.run(query_embedding=query_embedding, top_k=5)["documents"],
])["documents"][:3]

# 4. Reranked retrieval (vector candidates + pretrained cross-encoder)
candidates = vector_retriever.run(query_embedding=query_embedding, top_k=5)["documents"]
reranked_results = reranker.run(query=question, documents=candidates, top_k=3)["documents"]

# Display the top results from each method
for method, results in {
    "Vector RAG": vector_results,
    "Keyword RAG (BM25)": bm25_results,
    "Hybrid RAG": hybrid_results,
    "Reranked RAG": reranked_results,
}.items():
    print(f"\n{method}")
    for rank, doc in enumerate(results, start=1):
        print(f"  {rank}. {doc.meta['title']}  (score={doc.score:.3f})")
        print(f"     {doc.content}")
